# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. My rule and its reason codes

## Rule

Prioritize content for refresh when it has **meaningful search visibility but weak organic clicks**, indicating that the content still has search demand but may have an opportunity to improve its performance.

The baseline score increases when a page has higher search impressions and a lower click-through rate (CTR). Pages with stronger evidence of search opportunity receive a higher refresh priority.

## Reason codes

* `HIGH_IMPRESSIONS_LOW_CTR` — the page receives meaningful search impressions but gets relatively few clicks.
* `LOW_SEARCH_SIGNAL` — the page has too little search visibility to justify prioritizing a refresh.

## Action labels

* `REFRESH` — prioritize the page for content review and potential refresh.
* `MONITOR` — keep the page under observation but do not prioritize it for immediate refresh.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
!pip -q install duckdb

from google.colab import userdata
import duckdb

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
""")

# Warehouse path
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test connection
con.sql("""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│   78835655 │
└────────────┘

In [12]:
import os
import numpy as np
import pandas as pd
# 1. Build March 2026 feature vector


queue = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# 2. Calculate CTR


queue["ctr"] = np.where(
    queue["gsc_impressions"] > 0,
    queue["gsc_clicks"] / queue["gsc_impressions"],
    0
)

# 3. Define the two signals


# Use a practical minimum volume instead of the median.
# At least 100 impressions means the page has meaningful
# search visibility for this simple baseline.

MIN_IMPRESSIONS = 100

# Pages with CTR below 2% are treated as having a
# potential click opportunity.
MAX_CTR = 0.02

high_impressions = (
    queue["gsc_impressions"] >= MIN_IMPRESSIONS
)

low_ctr = (
    queue["ctr"] < MAX_CTR
)

# 4. Define the refresh opportunity

refresh_opportunity = (
    high_impressions &
    low_ctr
)

# 5. Calculate score

# Only pages satisfying the rule receive a positive
# refresh-opportunity score.

queue["score"] = np.where(
    refresh_opportunity,

    # More impressions = more potential opportunity.
    # Lower CTR = greater potential click improvement.
    np.log1p(queue["gsc_impressions"])
    * (1 - queue["ctr"].clip(0, 1)),

    0
)


# 6. Reason code

queue["reason_code"] = np.where(
    refresh_opportunity,
    "HIGH_IMPRESSIONS_LOW_CTR",
    "LOW_SEARCH_SIGNAL"
)



# 7. Action label


queue["action"] = np.where(
    refresh_opportunity,
    "REFRESH",
    "MONITOR"
)

# 8. Rank
queue = queue.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

# 9. Final ranked queue


baseline_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
]

# 10. Write CSV
os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = (
    "work/outputs/baseline_action_score.csv"
)

baseline_queue.to_csv(
    output_path,
    index=False
)

# 11. Verification

print("=" * 60)
print("BASELINE QUEUE CREATED")
print("=" * 60)

print(f"Saved: {output_path}")
print(f"Rows: {len(baseline_queue):,}")

print()
print("RULE THRESHOLDS")
print("-" * 60)
print(f"Minimum impressions: {MIN_IMPRESSIONS}")
print(f"Maximum CTR: {MAX_CTR:.2%}")

print()
print("ACTION COUNTS")
print("-" * 60)
print(baseline_queue["action"].value_counts())

print()
print("REASON CODE COUNTS")
print("-" * 60)
print(baseline_queue["reason_code"].value_counts())

print()
print("TOP 10")
print("-" * 60)
display(baseline_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BASELINE QUEUE CREATED
Saved: work/outputs/baseline_action_score.csv
Rows: 331,437

RULE THRESHOLDS
------------------------------------------------------------
Minimum impressions: 100
Maximum CTR: 2.00%

ACTION COUNTS
------------------------------------------------------------
action
MONITOR    230777
REFRESH    100660
Name: count, dtype: int64

REASON CODE COUNTS
------------------------------------------------------------
reason_code
LOW_SEARCH_SIGNAL           230777
HIGH_IMPRESSIONS_LOW_CTR    100660
Name: count, dtype: int64

TOP 10
------------------------------------------------------------


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH


In [13]:
baseline_queue["action"].value_counts()

,count
action,
MONITOR,230777
REFRESH,100660


In [14]:
baseline_queue.head(10)

,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH


## 3. Top-20 review

The top 20 rows are reviewed as decision-support recommendations, not as guaranteed refresh candidates.

Each row includes the action and reason code produced by the baseline rule, a confidence note, and a statement of what could make the recommendation wrong.

The baseline is intentionally simple: high search impressions combined with low CTR is treated as a refresh opportunity. A skeptic should still verify search intent, SERP context, data availability, and other business factors before taking action.

In [15]:
top20 = baseline_queue.head(20).copy()

top20


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH


In [16]:
top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = (
    "Moderate confidence: strong search visibility with low CTR under the baseline rule."
)

top20_review["what_would_make_it_wrong"] = (
    "The recommendation could be wrong if the low CTR reflects search intent, "
    "SERP features, brand behavior, or data-quality/availability issues rather than "
    "content needing a refresh."
)

top20_review

,rank,client_hash_id,content_hash_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH,Moderate confidence: strong search visibility ...,The recommendation could be wrong if the low C...


## 4. Weak picks + leakage check

The top-ranked recommendations are not guaranteed refresh opportunities. A weak pick would be a page where high impressions and low CTR have a plausible explanation unrelated to content quality, such as search intent mismatch, SERP features, branded-query behavior, or incomplete data.

The baseline uses only March 2026 historical features. It does not use product flags, future-window metrics, or label-derived fields.

Potentially weak picks should therefore be treated as decision-support candidates requiring human review rather than automatic refresh decisions.

In [17]:
# Leakage check: confirm the baseline queue contains only allowed fields

allowed_input_features = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
}

queue_input_columns = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
}

product_flag_keywords = [
    "flag",
    "product",
    "quick_win",
    "refresh",
    "opportunity",
    "label",
    "outcome",
    "future"
]

# Check that all model inputs are allowed historical features
unexpected_inputs = queue_input_columns - allowed_input_features

# Check for suspicious product/future/label fields in the queue
suspicious_columns = [
    col for col in baseline_queue.columns
    if any(keyword in col.lower() for keyword in product_flag_keywords)
    and col not in ["action", "reason_code"]
]

print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print("Unexpected input features:", unexpected_inputs)
print("Suspicious product/future/label columns:", suspicious_columns)

if not unexpected_inputs and not suspicious_columns:
    print("\nRESULT: PASS")
    print("No product flags, future-window fields, or label-derived inputs are used.")
else:
    print("\nRESULT: REVIEW REQUIRED")


LEAKAGE CHECK
Unexpected input features: set()
Suspicious product/future/label columns: []

RESULT: PASS
No product flags, future-window fields, or label-derived inputs are used.


**Leakage verdict: PASS.**

The baseline score uses only historical March 2026 search and analytics features. No future-window measurements, product flags, or label-derived fields are used as inputs. The action and reason code are outputs of the rule, not input features.

## Self-check

- Every section above is completed with both the required explanation and supporting code.
- The notebook runs from top to bottom without errors using **Runtime → Run all**.
- No client names, URLs, or private queries are included.
- Claims are written using careful terms such as **observed, measured, directional,** and **decision-support**.
- The completed notebook is committed under `work/notebooks/w04_baseline_score.ipynb`.
- The baseline queue is generated at `work/outputs/baseline_action_score.csv`.
- The baseline uses only historical features and does not use future-window, product-flag, or label-derived inputs.